# Ollama + Ngrok Agent Setup

This notebook updates Ollama, starts a public Ngrok tunnel using `.env`, pulls the newest versioned official Llama model, and lists likely agent-ready models from the official Ollama library. It never downloads catalogue models automatically.

Run cell 5 to open the catalogue. You can browse without downloading Llama first. After running cell 3, use Start Ollama or Stop Ollama to control its server. Type a model name in Search and press Enter. Use Previous, Next, or Go to navigate and Agent ready only to filter. Refresh in the Model column header reloads the catalogue. The row icons download a model's default tag or select an installed tag for confirmation. Use the top Download controls to enter a specific tag.

After Ollama is running, open cell 6 to chat with your downloaded models. Each reply shows its speed, token counts, and estimated context use when Ollama reports the needed metrics. Use Compact to summarize older turns and Attach files to add text or supported media. Downloads and uninstalls from cell 5 refresh the current catalogue page and any open chat picker. Use Refresh models for changes made elsewhere. To diagnose a chat or model-list failure, turn on Debug and retry the action.

ngrok starts automatically when configured and retries while this kernel is running. Use Offline ngrok to stop the tunnel and retries, or Online ngrok to resume. Use the current URL shown in the ngrok controls.

In Google Colab, the setup cell prompts you to upload `.env` when it is not already in `/content`. It also checks an existing mounted Drive copy at `/content/drive/MyDrive/Ollama-Ngrok/.env`:

```env
NGROK_AUTHTOKEN=your_ngrok_authtoken
```

## 1. Stop an existing ngrok tunnel (optional)

Run this on the runtime whose ngrok tunnel you want to stop.

In [ ]:
if 'ngrok_controller' in globals():
    ngrok_controller.stop()
    print(ngrok_controller.status)
else:
    from pyngrok import ngrok
    ngrok.kill()

## 2. Check the Colab GPU (optional)

In [1]:
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-f1d07a27-8b96-6138-4a8a-6c12b2423704)


## 3. Install and start Ollama and ngrok

In [10]:
import os
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path
from urllib.request import Request, urlopen

def run(command, *, check=True, **kwargs):
    return subprocess.run(command, check=check, text=True, **kwargs)

def load_env(path):
    if not path.is_file():
        return False
    for raw_line in path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip(chr(34)).strip(chr(39)))
    return True

def find_env_file():
    candidates = (
        Path('/content/.env'),
        Path('/content/drive/MyDrive/Ollama-Ngrok/.env'),
        Path('.env'),
    )
    return next((path for path in candidates if path.is_file()), None)

def upload_colab_env():
    try:
        from google.colab import files
    except ImportError:
        return None
    print('Upload your local .env file to configure Ngrok for this Colab runtime.')
    uploaded = files.upload()
    contents = uploaded.get('.env')
    if contents is None:
        print('No .env file was uploaded; Ngrok will not start.')
        return None
    path = Path('/content/.env')
    path.write_bytes(contents)
    return path

# Ollama's Linux release archives can require the zstd executable.
if sys.platform.startswith('linux') and shutil.which('zstd') is None:
    if shutil.which('apt-get') is None:
        raise RuntimeError('Install zstd with your system package manager, then rerun this cell.')
    apt_command = ['apt-get']
    if os.geteuid() != 0:
        if shutil.which('sudo') is None:
            raise RuntimeError('Ask your runtime administrator to install zstd, then rerun this cell.')
        apt_command = ['sudo', '-n', *apt_command]
    print('Installing zstd for Ollama archive extraction...')
    try:
        run([*apt_command, 'update', '-qq'])
        run([*apt_command, 'install', '-y', '-qq', 'zstd'])
    except subprocess.CalledProcessError as error:
        raise RuntimeError('Could not install zstd. Check the apt or sudo output above, then rerun this cell.') from error
    if shutil.which('zstd') is None:
        raise RuntimeError('zstd is still missing from PATH after installation.')

print('Installing or updating Ollama...')
ollama_install_url = 'https://' + 'ollama.com' + '/install.sh'
installer_download = subprocess.run(
    ['curl', '-fsSL', ollama_install_url], capture_output=True, text=True
)
if installer_download.returncode != 0:
    raise RuntimeError(f'Could not download the Ollama installer: {installer_download.stderr.strip()}')
installer_run = subprocess.run(
    ['bash', '-s'], input=installer_download.stdout, capture_output=True, text=True
)
if installer_run.stdout.strip():
    print(installer_run.stdout.strip())
if installer_run.returncode != 0:
    error_output = installer_run.stderr.strip() or 'The installer returned no error output.'
    raise RuntimeError(f'Ollama installer failed (exit {installer_run.returncode}): {error_output}')

os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
os.environ['OLLAMA_ORIGINS'] = '*'
ollama_server = globals().get('ollama_server')

def ollama_server_running():
    return ollama_server is not None and ollama_server.poll() is None

def stop_ollama_server():
    if not ollama_server_running():
        return
    ollama_server.terminate()
    try:
        ollama_server.wait(timeout=10)
    except subprocess.TimeoutExpired as error:
        raise RuntimeError('Ollama is still stopping. Check ollama.log and try again if needed.') from error

def start_ollama_server():
    global ollama_server
    if not ollama_server_running():
        with open('ollama.log', 'a', encoding='utf-8') as log_file:
            ollama_server = subprocess.Popen(['ollama', 'serve'], stdout=log_file, stderr=log_file)
    try:
        deadline = time.monotonic() + 20
        while ollama_server_running() and time.monotonic() < deadline:
            try:
                with urlopen('http://127.0.0.1:11434/api/version', timeout=1):
                    if ollama_server_running():
                        return
            except OSError:
                time.sleep(0.25)
    except KeyboardInterrupt:
        stop_ollama_server()
        raise
    stop_ollama_server()
    raise RuntimeError('Ollama did not become ready. Check ollama.log for startup errors.')

if ollama_server is None:
    run(['bash', '-lc', 'pkill ollama || true'], check=False)
else:
    stop_ollama_server()
start_ollama_server()
run(['ollama', 'list'])

run([sys.executable, '-m', 'pip', 'install', '--quiet', 'pyngrok'])
from html import escape
from threading import Event, Lock, Thread
from urllib.error import HTTPError

from pyngrok import conf, ngrok, process
from pyngrok.exception import PyngrokError


class NgrokEndpoint:
    upstream_url = 'http://127.0.0.1:11434'

    def __init__(self):
        self.config = conf.PyngrokConfig()
        self.config.auth_token = None
        self.public_url = None
        self.status = 'Offline. Set NGROK_AUTHTOKEN in .env and rerun setup.'
        self._lock = Lock()
        self._stop = Event()
        self._stop.set()
        self._controls = None

    def configure(self, token, *, auto_start=True):
        if token == self.config.auth_token:
            return
        should_start = not self._stop.is_set() or not self.config.auth_token
        if self.config.auth_token and not self.stop():
            return
        self.config.auth_token = token
        if token and should_start and auto_start:
            self.start()
        else:
            self._set_status('Offline. Click Online ngrok to reconnect.' if token
                             else 'Offline. Set NGROK_AUTHTOKEN in .env and rerun setup.')

    def start(self):
        with self._lock:
            if not self._stop.is_set() or not self.config.auth_token:
                return
            self._stop = Event()
            self._set_status('Connecting…')
            try:
                self._check()
                Thread(target=self._monitor, args=(self._stop,),
                       name='ngrok-monitor', daemon=True).start()
            except (KeyboardInterrupt, RuntimeError):
                self._stop.set()
                self._set_status('Automatic reconnect stopped. Click Online ngrok to retry.')
                raise

    def stop(self):
        with self._lock:
            self._stop.set()
            try:
                ngrok.kill(pyngrok_config=self.config)
            except (PyngrokError, OSError):
                self._set_status('Could not stop the ngrok agent. Click Offline ngrok to retry.')
                return False
            self.public_url = None
            self._set_status('Offline. Automatic reconnect is off.')
            return True

    def _monitor(self, stopped):
        delay = 30
        while not stopped.wait(delay):
            with self._lock:
                if stopped.is_set():
                    return
                healthy = self._check()
            delay = 30 if healthy else min(delay * 2, 120)

    def _check(self):
        try:
            if self.public_url and process.is_process_running(self.config.ngrok_path):
                request = Request(f'{self.public_url.rstrip("/")}/api/version',
                                  headers={'ngrok-skip-browser-warning': '1'})
                try:
                    with urlopen(request, timeout=5) as response:
                        if response.headers.get_content_type() == 'application/json':
                            self._set_status('Online.')
                            return True
                    self._set_status('The endpoint responded. Ollama API health is unverified.')
                    return False
                except HTTPError as error:
                    with error:
                        code = error.headers.get('ngrok-error-code', '')
                    if code in {'ERR_NGROK_8012', '8012'}:
                        self._set_status(f'Cannot reach Ollama at {self.upstream_url}. Check ollama.log. Retrying automatically.')
                        return False
                    if code not in {'ERR_NGROK_3200', '3200'}:
                        self._set_status(f'Public API returned HTTP {error.code}. Checking again shortly.')
                        return False
                    self._set_status('ngrok reported the endpoint offline. Reconnecting…')
                    ngrok.kill(pyngrok_config=self.config)
            self.public_url = None
            tunnel = ngrok.connect(self.upstream_url, pyngrok_config=self.config)
            self.public_url = tunnel.public_url
            self._set_status('Tunnel started. Checking the public API shortly.')
            return True
        except (PyngrokError, OSError, ValueError) as error:
            if 'ERR_NGROK_334' in str(error):
                self._set_status('This domain is in use. Stop the old ngrok session. Retrying automatically.')
            else:
                self._set_status('ngrok or the public API could not be reached. Retrying automatically.')
            return False

    def _set_status(self, message):
        self.status = message
        if self._controls is None:
            return
        url = self.public_url or ''
        link = ''
        if url.startswith(('https://', 'http://')):
            safe_url = escape(url)
            link = f' <a href="{safe_url}" target="_blank" rel="noopener noreferrer">{safe_url}</a>'
        self._label.value = f'<b>ngrok</b> {escape(message)}{link}'
        self._online_button.disabled = not self.config.auth_token or not self._stop.is_set()
        self._offline_button.disabled = self._stop.is_set() and not self.public_url and not message.startswith('Could not stop')

    def controls(self):
        import ipywidgets as widgets

        with self._lock:
            if self._controls is None:
                self._label = widgets.HTML()
                self._online_button = widgets.Button(description='Online ngrok', button_style='success')
                self._offline_button = widgets.Button(description='Offline ngrok', button_style='danger')
                self._online_button.on_click(lambda _: self.start())
                self._offline_button.on_click(lambda _: self.stop())
                buttons = widgets.HBox([self._online_button, self._offline_button])
                self._controls = widgets.VBox([self._label, buttons])
                self._set_status(self.status)
            return self._controls


env_file = find_env_file() or upload_colab_env()
if env_file:
    load_env(env_file)
ngrok_token = os.environ.get('NGROK_AUTHTOKEN') or os.environ.get('NGROK_TOKEN') or None
auto_start_ngrok = True
if 'ngrok_controller' not in globals():
    if ngrok_token and process.is_process_running(conf.get_default().ngrok_path):
        ngrok.kill()
    ngrok_controller = NgrokEndpoint()
elif getattr(ngrok_controller, 'upstream_url', None) != NgrokEndpoint.upstream_url:
    # Replace a port-only tunnel from an older setup run and honor a manual stop.
    auto_start_ngrok = not ngrok_controller.config.auth_token or not ngrok_controller._stop.is_set()
    if not ngrok_controller.stop():
        raise RuntimeError('Could not replace the existing ngrok tunnel. Click Offline ngrok and rerun setup.')
    ngrok_controller = NgrokEndpoint()
ngrok_controller.configure(ngrok_token, auto_start=auto_start_ngrok)
print(f'ngrok: {ngrok_controller.status}')
if ngrok_controller.public_url:
    print(f'Ngrok URL: {ngrok_controller.public_url}')
try:
    from IPython.display import display

    display(ngrok_controller.controls())
except ImportError:
    print('Use ngrok_controller.start() or ngrok_controller.stop() to control ngrok.')


Installing or updating Ollama...
ngrok: Tunnel started. Checking the public API shortly.
Ngrok URL: https://manor-sniff-magical.ngrok-free.dev


## 4. Find and download the latest Llama model

In [5]:
from functools import lru_cache
from html import unescape
from urllib.request import Request, urlopen
from urllib.parse import unquote

def fetch_text(url, timeout=30):
    request = Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urlopen(request, timeout=timeout) as response:
        return response.read().decode('utf-8', errors='replace')

@lru_cache(maxsize=1)
def library_models():
    html = fetch_text('https://ollama.com/library')
    slugs = re.findall(r'''href\s*=\s*["']/library/([^"']+)["']''', html)
    names = (unquote(slug) for slug in slugs)
    models = tuple(dict.fromkeys(name for name in names if re.fullmatch(r'[a-zA-Z0-9][a-zA-Z0-9._-]*', name)))
    if not models:
        raise RuntimeError('No model links found in the Ollama library index. Catalogue unavailable.')
    return models

def newest_llama_model():
    candidates = []
    for name in library_models():
        match = re.fullmatch(r'llama(\d+(?:\.\d+)*)', name, flags=re.IGNORECASE)
        if match:
            version = tuple(int(part) for part in match.group(1).split('.'))
            candidates.append((version, name))
    if not candidates:
        raise RuntimeError('Could not find a versioned Llama model in the official Ollama library.')
    return max(candidates)[1]

LATEST_LLAMA_MODEL = newest_llama_model()
print(f'Pulling newest versioned Llama model: {LATEST_LLAMA_MODEL}')
run(['ollama', 'pull', LATEST_LLAMA_MODEL])
print(f'{LATEST_LLAMA_MODEL} is ready.')


Pulling newest versioned Llama model: llama4


KeyboardInterrupt: 

## 5. Open the interactive model catalogue

In [11]:
import json
import re
from functools import lru_cache
from html import unescape
from http.client import HTTPException
from urllib.error import HTTPError, URLError
from urllib.parse import unquote
from urllib.request import Request, urlopen

# Reuse the model-download cell's index when it has already run.
if 'fetch_text' not in globals() or 'library_models' not in globals():
    def fetch_text(url, timeout=30):
        request = Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urlopen(request, timeout=timeout) as response:
            return response.read().decode('utf-8', errors='replace')

    @lru_cache(maxsize=1)
    def library_models():
        html = fetch_text('https://ollama.com/library')
        slugs = re.findall(r'''href\s*=\s*["']/library/([^"']+)["']''', html)
        names = (unquote(slug) for slug in slugs)
        models = tuple(dict.fromkeys(name for name in names if re.fullmatch(r'[a-zA-Z0-9][a-zA-Z0-9._-]*', name)))
        if not models:
            raise RuntimeError('No model links found in the Ollama library index. Catalogue unavailable.')
        return models

# This is a catalogue filter, not a guarantee that a model will work with every agent framework.
# Raise or lower MIN_CONTEXT_K / MIN_AGENT_SIGNALS to tune the result.
MODELS_PER_PAGE = 20
MIN_CONTEXT_K = 32
MIN_AGENT_SIGNALS = 2
AGENT_SIGNALS = ('tool', 'function call', 'agent', 'reasoning', 'coder', 'code', 'json')

def local_model_request(action, *, method='GET', payload=None, timeout=10):
    data = json.dumps(payload).encode('utf-8') if payload is not None else None
    request = Request(f'http://127.0.0.1:11434/api/{action}', data=data, method=method,
                      headers={'Content-Type': 'application/json'})
    try:
        return urlopen(request, timeout=timeout)
    except HTTPError as error:
        try:
            with error:
                body = json.load(error)
        except (OSError, ValueError):
            body = {}
        detail = body.get('error') if isinstance(body, dict) else None
        raise RuntimeError(detail or f'Ollama returned HTTP {error.code}.') from error

def installed_model_tags():
    with local_model_request('tags') as response:
        payload = json.load(response)
    models = payload.get('models') if isinstance(payload, dict) else None
    if not isinstance(models, list):
        raise ValueError('Ollama returned an invalid model list.')
    names = (item.get('name') or item.get('model') for item in models if isinstance(item, dict))
    return sorted({name for name in names if isinstance(name, str) and name}, key=str.lower)

def installed_models():
    try:
        return {tag.rsplit(':', 1)[0] for tag in installed_model_tags()}
    except (OSError, RuntimeError, ValueError):
        return set()

@lru_cache(maxsize=None)
def catalogue_metadata(model):
    html = fetch_text(f'https://ollama.com/library/{model}')
    description_match = re.search(r'<meta name="description" content="([^"]*)"', html)
    description = unescape(description_match.group(1)) if description_match else ''
    tags_text = re.sub(r'<[^>]+>', ' ', fetch_text(f'https://ollama.com/library/{model}/tags'))
    context_match = re.search(r'(\d+(?:\.\d+)?)\s*K\s*context', tags_text, flags=re.IGNORECASE)
    size_match = re.search(r'([0-9.]+\s?(?:GB|MB))', tags_text, flags=re.IGNORECASE)
    context_k = float(context_match.group(1)) if context_match else 0
    signals = [signal for signal in AGENT_SIGNALS if signal in f'{model} {description}'.lower()]
    size = size_match.group(1) if size_match else '---'
    return context_k, size, signals, description

def catalogue_row(model):
    try:
        context_k, size, signals, description = catalogue_metadata(model)
    except (OSError, ValueError) as error:
        return model, 'METADATA ERROR', '---', 0, [], str(error)
    missing = []
    if context_k < MIN_CONTEXT_K:
        missing.append(f'context < {MIN_CONTEXT_K}K')
    if len(signals) < MIN_AGENT_SIGNALS:
        missing.append(f'agent signals < {MIN_AGENT_SIGNALS}')
    status = 'AGENT READY' if not missing else 'NOT READY: ' + ', '.join(missing)
    return model, status, size, context_k, signals, description

def catalogue_page_data(page, *, search='', only_agent_ready=False, refresh=False):
    if page < 1 or MODELS_PER_PAGE < 1:
        raise ValueError('Use a positive page number and page size.')
    if refresh:
        library_models.cache_clear()
        catalogue_metadata.cache_clear()
    models = sorted(library_models(), key=str.lower)
    index_count = len(models)
    query = search.strip().casefold()
    if query:
        models = [model for model in models if query in model.casefold()]
    start = (page - 1) * MODELS_PER_PAGE
    stop = start + MODELS_PER_PAGE
    errors = 0

    if only_agent_ready:
        scanned = [catalogue_row(model) for model in models]
        eligible = [row for row in scanned if row[1] == 'AGENT READY']
        errors = sum(row[1] == 'METADATA ERROR' for row in scanned)
        rows = eligible[start:stop]
        total = len(eligible)
    else:
        rows = [catalogue_row(model) for model in models[start:stop]]
        total = len(models)

    pages = (total + MODELS_PER_PAGE - 1) // MODELS_PER_PAGE
    return rows, total, pages, index_count, errors

def show_catalogue_page(page=1, *, search='', only_agent_ready=False, refresh=False):
    if page < 1 or MODELS_PER_PAGE < 1:
        print('Use a positive page number and page size.')
        return
    rows, total, pages, index_count, errors = catalogue_page_data(
        page, search=search, only_agent_ready=only_agent_ready, refresh=refresh
    )
    if errors:
        print(f'Partial agent-ready results: metadata unavailable for {errors} model(s).')
    if not pages:
        print('No models matched the current filters.')
        return
    if page > pages:
        print(f'Choose a page from 1 to {pages}.')
        return

    installed = installed_models()
    print(f'Page {page} of {pages} | Matching models: {total} | Official index: {index_count}')
    print('Agent readiness is based on catalogue metadata. The listing covers model families.')
    for model, status, size, context_k, signals, description in rows:
        local = 'INSTALLED' if model in installed else 'NOT INSTALLED'
        print('─' * 120)
        print(f'MODEL: {model} | STATUS: {status} | LOCAL: {local}')
        print(f"SOURCE: Official Ollama | SIZE: {size} | CONTEXT: {context_k:.0f}K | SIGNALS: {', '.join(signals) or 'none'}")
        print(f"DESCRIPTION: {description or '---'}")
        print(f'URL: https://ollama.com/library/{model}')


def open_model_catalogue():
    try:
        import ipywidgets as widgets
    except ImportError:
        print('To enable catalogue controls, run %pip install ipywidgets in a separate cell and rerun this cell.')
        show_catalogue_page(page=1)
        return

    from html import escape
    from IPython.display import display

    title = widgets.HTML('<h3>Ollama Model Catalogue</h3>')
    summary = widgets.HTML()
    table = widgets.GridBox(layout=widgets.Layout(
        grid_template_columns='minmax(205px, 1.2fr) 95px minmax(120px, 1fr) 65px 70px minmax(90px, 1fr) minmax(200px, 3fr)',
        grid_gap='4px 8px', align_items='center', overflow='auto', width='100%',
    ))
    row_layout = widgets.Layout(align_items='center')
    icon_layout = widgets.Layout(width='34px', flex='0 0 34px')
    text_layout = widgets.Layout(min_width='0', margin='4px 0')
    search = widgets.Text(description='Search', placeholder='Model name, e.g. llama',
                          continuous_update=False, style={'description_width': 'initial'},
                          layout=widgets.Layout(width='235px'))
    previous = widgets.Button(description='Previous')
    next_page = widgets.Button(description='Next')
    page_input = widgets.BoundedIntText(value=1, min=1, max=1, description='Page',
                                        layout=widgets.Layout(width='140px'))
    go = widgets.Button(description='Go', layout=widgets.Layout(width='55px'))
    ready_only = widgets.Checkbox(description='Agent ready only', indent=False)
    refresh_button = widgets.Button(description='Refresh', icon='refresh',
                                    tooltip='Reload the catalogue and installed models',
                                    layout=widgets.Layout(width='90px'))
    model_header = widgets.HBox([widgets.HTML('<b>Model</b>', layout=text_layout), refresh_button],
                               layout=row_layout)
    columns = ('Local status', 'Agent readiness', 'Size', 'Context', 'Signals', 'Description')
    headers = (model_header, *(widgets.HTML(f'<b>{column}</b>', layout=text_layout) for column in columns))
    table.children = headers
    navigation = [search, previous, next_page, page_input, go, ready_only]
    controls = [*navigation, refresh_button]
    current_page = 1
    page_count = 0

    download_picker = widgets.Dropdown(description='On this page')
    download_tag = widgets.Text(description='Tag', placeholder='Optional, e.g. 1b')
    download = widgets.Button(description='Download', button_style='primary')
    installed_picker = widgets.Dropdown(description='Installed')
    confirm_uninstall = widgets.Checkbox(description='Confirm uninstall', indent=False)
    uninstall = widgets.Button(description='Uninstall', button_style='danger')
    start_ollama = widgets.Button(description='Start Ollama', icon='play', button_style='success')
    stop_ollama = widgets.Button(description='Stop Ollama', icon='stop', button_style='danger')
    local_status = widgets.Label()
    operation_status = widgets.Label()
    download_details = widgets.HTML()
    visible_rows = []
    rendered_rows = None
    row_widgets = []
    row_actions = {}
    tags_by_family = {}
    local_tags = None
    has_model_snapshot = False
    busy = False
    start_server = globals().get('start_ollama_server')
    stop_server = globals().get('stop_ollama_server')
    server_running = globals().get('ollama_server_running')
    can_control_server = all(callable(action) for action in (start_server, stop_server, server_running))

    def sync_action_buttons(_=None):
        unavailable = busy or local_tags is None
        managed_server_running = can_control_server and server_running()
        start_ollama.disabled = busy or not can_control_server or managed_server_running or local_tags is not None
        stop_ollama.disabled = busy or not managed_server_running
        download_picker.disabled = download_tag.disabled = unavailable or not download_picker.options
        download.disabled = unavailable or not download_picker.value
        installed_picker.disabled = unavailable or not local_tags
        confirm_uninstall.disabled = unavailable or not installed_picker.value
        uninstall.disabled = confirm_uninstall.disabled or not confirm_uninstall.value
        for family, (get_model, remove_model, _) in row_actions.items():
            get_model.disabled = unavailable
            remove_model.disabled = unavailable or family not in tags_by_family

    def reload_local_models():
        nonlocal local_tags, has_model_snapshot
        selected = installed_picker.value
        try:
            local_tags = installed_model_tags()
        except (OSError, RuntimeError, ValueError) as error:
            local_tags = None
            if can_control_server and not server_running():
                local_status.value = 'Ollama offline. Click Start Ollama to update model status.'
            else:
                cause = error.reason if isinstance(error, URLError) and isinstance(error.reason, OSError) else error
                local_status.value = f'Could not read the local model list ({type(cause).__name__}). Check ollama.log, then click Refresh.'
            if has_model_snapshot:
                local_status.value += ' Showing last checked status.'
        else:
            has_model_snapshot = True
            local_status.value = f'Ollama online. {len(local_tags)} installed model tag(s).'
            tags_by_family.clear()
            for tag in local_tags:
                tags_by_family.setdefault(tag.rsplit(':', 1)[0], []).append(tag)
        if not can_control_server:
            local_status.value += ' Run Cell 3, then Cell 5, to enable the server buttons.'
        installed_picker.options = local_tags or []
        installed_picker.value = selected if selected in installed_picker.options else next(iter(installed_picker.options), None)
        confirm_uninstall.value = False
        render_model_table()
        sync_action_buttons()

    def change_server(starting):
        nonlocal busy
        button = start_ollama if starting else stop_ollama
        if busy or button.disabled:
            return
        busy = True
        for control in controls:
            control.disabled = True
        sync_action_buttons()
        operation_status.value = 'Starting Ollama…' if starting else 'Stopping Ollama…'
        try:
            action = start_server if starting else stop_server
            action()
            operation_status.value = 'Ollama started.' if starting else 'Ollama stopped.'
        except (OSError, RuntimeError) as error:
            operation_status.value = f'{button.description} failed. {error}'
        except KeyboardInterrupt:
            operation_status.value = f'{button.description} interrupted.'
        finally:
            busy = False
            reload_local_models()
            sync_navigation()
        refresh_chat_models = globals().get('_refresh_downloaded_model_chat')
        if callable(refresh_chat_models):
            refresh_chat_models(tags=local_tags)

    pull_stages = {
        'pulling manifest': 'Fetching manifest',
        'verifying sha256 digest': 'Verifying SHA-256 digest',
        'writing manifest': 'Writing manifest',
        'removing any unused layers': 'Removing unused layers',
    }

    def format_download_size(count):
        unit = 'B'
        for next_unit in ('KiB', 'MiB', 'GiB', 'TiB'):
            if count < 1024:
                break
            count /= 1024
            unit = next_unit
        return f'{count:d} {unit}' if unit == 'B' else f'{count:.2f} {unit}'

    def stream_pull(name, timeout, report):
        payload = {'model': name, 'stream': True}
        with local_model_request('pull', method='POST', payload=payload, timeout=timeout) as response:
            for line in response:
                if not line.strip():
                    continue
                progress = json.loads(line)
                if not isinstance(progress, dict):
                    raise ValueError('Ollama returned invalid download progress.')
                status = progress.get('status')
                if isinstance(status, str) and status.strip() and status not in {'success', 'error'}:
                    stage = pull_stages.get(status, status)
                    if status != 'pulling manifest' and status.startswith('pulling '):
                        stage = f'Downloading layer {status.removeprefix("pulling ")}'
                    detail = ''
                    total, completed = progress.get('total'), progress.get('completed', 0)
                    if isinstance(total, int) and total > 0 and isinstance(completed, int):
                        completed = min(total, max(0, completed))
                        detail = (f'{format_download_size(completed)} / {format_download_size(total)} '
                                  f'({completed * 100 // total}% of layer)')
                    report(stage, detail)
                if progress.get('error'):
                    raise RuntimeError(progress['error'])
                if status == 'success':
                    return
        raise RuntimeError('The progress stream ended before Ollama reported success. Check ollama.log for details.')

    def pull_model(name):
        stages = []

        def report(stage, detail=''):
            entry = (stage, detail)
            if stages and stages[-1][0] == stage:
                if stages[-1] == entry:
                    return
                stages[-1] = entry
            else:
                stages.append(entry)
            operation_status.value = ' · '.join(part for part in (name, stage, detail) if part)
            history = ''.join(f'<li><b>{escape(label)}</b> {escape(text)}</li>'
                              for label, text in stages)
            download_details.value = (f'<b>{escape(name)} download stages</b>'
                                      f'<ul style="margin:0.25em 0;overflow-wrap:anywhere">{history}</ul>')

        for attempt, timeout in enumerate((120, 300), start=1):
            report('Requesting pull', f'Attempt {attempt} of 2')
            try:
                stream_pull(name, timeout, report)
            except (OSError, RuntimeError, ValueError, HTTPException) as error:
                stage = stages[-1][0]
                reason = str(error) or type(error).__name__
                if isinstance(getattr(error, 'reason', error), TimeoutError):
                    reason = f'Timed out waiting for Ollama ({timeout}s). {reason}'
                    if attempt == 1:
                        report('Retrying after timeout', f'Last stage was {stage}. {reason}')
                        continue
                    reason += ' Click Download again in this runtime to resume.'
                failure = f'Last stage was {stage}. {reason}'
                report('Download failed', failure)
                raise RuntimeError(failure) from error
            except KeyboardInterrupt:
                report('Download interrupted', f'Last stage was {stages[-1][0]}.')
                raise
            else:
                report('Download complete')
                return

    def change_model(action, name):
        nonlocal busy
        if busy or local_tags is None:
            return
        busy = True
        for control in controls:
            control.disabled = True
        sync_action_buttons()
        operation_status.value = f'{action} requested for {name}…'
        changed = False
        try:
            if action == 'Download':
                pull_model(name)
                operation_status.value = f'Downloaded {name}.'
            else:
                with local_model_request('delete', method='DELETE', payload={'model': name}):
                    pass
                operation_status.value = f'Uninstalled {name}.'
        except (OSError, RuntimeError, ValueError) as error:
            operation_status.value = f'{action} failed for {name}. {error}'
        except KeyboardInterrupt:
            operation_status.value = f'Stopped waiting for {action.lower()} of {name}.'
        else:
            changed = True
        finally:
            busy = False
            reload_local_models()
            sync_navigation()
        refresh_chat_models = globals().get('_refresh_downloaded_model_chat')
        if changed and callable(refresh_chat_models):
            refresh_chat_models(tags=local_tags)

    def download_selected(_):
        family = download_picker.value
        if family not in download_picker.options:
            return
        tag = download_tag.value.strip()
        if tag and not re.fullmatch(r'[a-zA-Z0-9][a-zA-Z0-9._-]*', tag):
            operation_status.value = 'Enter a tag such as 1b or leave the field empty for the default tag.'
            return
        name = f'{family}:{tag}' if tag else family
        change_model('Download', name)

    def uninstall_selected(_):
        selected = installed_picker.value
        if local_tags and selected in local_tags and confirm_uninstall.value:
            confirm_uninstall.value = False
            change_model('Uninstall', selected)

    def download_from_row(family):
        if family in row_actions:
            change_model('Download', family)

    def uninstall_from_row(family):
        choices = tags_by_family.get(family, [])
        if busy or local_tags is None or family not in row_actions or not choices:
            return
        if installed_picker.value not in choices:
            installed_picker.value = choices[0]
        confirm_uninstall.value = False
        operation_status.value = (
            f'Selected {installed_picker.value}. Check the tag above, then confirm and click Uninstall.'
        )
        sync_action_buttons()

    def render_model_table():
        nonlocal rendered_rows
        if rendered_rows is not visible_rows:
            old_widgets = row_widgets[:]
            row_widgets.clear()
            row_actions.clear()
            grid_items = list(headers)
            for model, status, size, context_k, signals, description in visible_rows:
                name = escape(model)
                link = widgets.HTML(
                    f'<a href="https://ollama.com/library/{name}" target="_blank" rel="noopener noreferrer" '
                    f'style="overflow-wrap:anywhere">{name}</a>', layout=text_layout,
                )
                get_model = widgets.Button(icon='download', tooltip=f'Download {model} (default tag)',
                                           layout=icon_layout)
                remove_model = widgets.Button(icon='trash', tooltip=f'Select an installed {model} tag to uninstall',
                                              layout=icon_layout)
                get_model.on_click(lambda _, family=model: download_from_row(family))
                remove_model.on_click(lambda _, family=model: uninstall_from_row(family))
                name_cell = widgets.HBox([link, get_model, remove_model], layout=row_layout)
                local_cell = widgets.HTML(layout=text_layout)
                row_actions[model] = (get_model, remove_model, local_cell)
                values = (status, size, f'{context_k:g}K' if context_k else '—',
                          ', '.join(signals) or '—', description or '—')
                details = [widgets.HTML(f'<div style="overflow-wrap:anywhere">{escape(value)}</div>',
                                        layout=text_layout) for value in values]
                grid_items.extend([name_cell, local_cell, *details])
                row_widgets.extend([link, get_model, remove_model, name_cell, local_cell, *details])
            table.children = tuple(grid_items)
            rendered_rows = visible_rows
            for widget in old_widgets:
                widget.close()
        for model, (_, _, local_cell) in row_actions.items():
            if not has_model_snapshot:
                local_cell.value = 'Unknown'
                continue
            status = 'Installed' if model in tags_by_family else 'Not installed'
            local_cell.value = f'{status} (last checked)' if local_tags is None else status

    def sync_navigation():
        for control in controls:
            control.disabled = False
        previous.disabled = page_count == 0 or current_page <= 1
        next_page.disabled = current_page >= page_count
        page_input.disabled = go.disabled = page_count <= 1

    def render(page=1, *, refresh=False):
        nonlocal current_page, page_count, visible_rows
        for control in controls:
            control.disabled = True
        summary.value = ('Checking agent readiness for matching model names. This can take a while.'
                         if ready_only.value else 'Loading catalogue page…')
        try:
            reload_local_models()
            visible_rows = []
            filters = {'search': search.value, 'only_agent_ready': ready_only.value}
            rows, total, pages, index_count, errors = catalogue_page_data(page, refresh=refresh, **filters)
            if pages and page > pages:
                page = pages
                rows, total, pages, index_count, errors = catalogue_page_data(page, **filters)
            visible_rows = rows
            current_page, page_count = page, pages
            page_input.max = max(1, pages)
            page_input.value = page
            summary.value = (
                f'<b>Page {page} of {pages}</b> · {total} matching models · {index_count} in the official index'
                if pages else f'<b>No models match the current filter.</b> · {index_count} in the official index'
            )
            if errors:
                summary.value += f'<br>Partial results: metadata unavailable for {errors} model(s).'
            if rows:
                summary.value += '<br>Agent readiness is estimated from catalogue metadata.'
        except (OSError, ValueError, RuntimeError) as error:
            page_count = 0
            summary.value = f'Could not load the catalogue. Use Refresh to retry. {escape(str(error))}'
        finally:
            selected = download_picker.value
            download_picker.options = [row[0] for row in visible_rows]
            download_picker.value = selected if selected in download_picker.options else next(iter(download_picker.options), None)
            render_model_table()
            sync_action_buttons()
            sync_navigation()

    def reset_uninstall_confirmation(_):
        confirm_uninstall.value = False
        sync_action_buttons()

    previous.on_click(lambda _: render(current_page - 1))
    next_page.on_click(lambda _: render(current_page + 1))
    go.on_click(lambda _: render(page_input.value))
    search.observe(lambda _: render(), names='value')
    ready_only.observe(lambda _: render(), names='value')
    refresh_button.on_click(lambda _: render(current_page, refresh=True))
    download_picker.observe(lambda _: setattr(download_tag, 'value', ''), names='value')
    installed_picker.observe(reset_uninstall_confirmation, names='value')
    confirm_uninstall.observe(sync_action_buttons, names='value')
    download.on_click(download_selected)
    uninstall.on_click(uninstall_selected)
    start_ollama.on_click(lambda _: change_server(True))
    stop_ollama.on_click(lambda _: change_server(False))
    layout = widgets.Layout(flex_flow='row wrap', align_items='center')
    toolbar = widgets.HBox(navigation, layout=layout)
    download_controls = widgets.HBox([download_picker, download_tag, download], layout=layout)
    uninstall_controls = widgets.HBox([installed_picker, confirm_uninstall, uninstall], layout=layout)
    sections = [title]
    if 'ngrok_controller' in globals():
        sections.append(ngrok_controller.controls())
    sections.append(widgets.HBox([start_ollama, stop_ollama, local_status], layout=layout))
    display(widgets.VBox([*sections, toolbar, download_controls, uninstall_controls,
                          operation_status, download_details, summary, table]))
    render()

open_model_catalogue()

## 6. Chat with downloaded models

In [7]:
def open_downloaded_model_chat():
    import base64
    import json
    import subprocess
    from collections import deque
    from html import escape
    from http.client import HTTPException
    from pathlib import Path
    from shutil import which
    from tempfile import TemporaryDirectory
    from time import monotonic
    from urllib.error import HTTPError, URLError
    from urllib.request import Request, urlopen

    try:
        import ipywidgets as widgets
        from IPython.display import display
    except ImportError:
        print('Run %pip install ipywidgets in a separate cell, then rerun this cell.')
        return

    model = widgets.Dropdown(description='Model', layout=widgets.Layout(width='100%'))
    refresh = widgets.Button(description='Refresh models')
    clear = widgets.Button(description='Clear chat')
    compact = widgets.Button(description='Compact', tooltip='Summarize older turns and keep the latest exchange')
    debug = widgets.Checkbox(description='Debug', indent=False)
    diagnostics = widgets.HTML(layout=widgets.Layout(display='none'))
    prompt = widgets.Textarea(placeholder='Type a message…', layout=widgets.Layout(width='100%'))
    send = widgets.Button(description='Send', button_style='primary')
    upload = widgets.FileUpload(description='Attach files', multiple=True)
    remove_files = widgets.Button(description='Remove files')
    attachment_status = widgets.Label()
    model_status = widgets.Label()
    turn_status = widgets.Label()
    status = widgets.Label()
    transcript = widgets.HTML()
    histories = {}
    transcripts = {}
    turn_stats = {}
    pending_files = []
    capabilities = set()
    busy = refreshing = False
    debug_entries = deque(maxlen=8)
    api_errors = (OSError, RuntimeError, ValueError, HTTPException)
    text_types = {'.txt', '.md', '.csv', '.tsv', '.json', '.jsonl', '.yaml', '.yml', '.toml',
                  '.log', '.py', '.js', '.ts', '.jsx', '.tsx', '.html', '.css', '.xml', '.sql', '.sh'}
    image_types = {'.jpg', '.jpeg', '.png', '.webp'}
    audio_types = {'.wav', '.mp3'}
    video_formats = {'.mp4': 'mov', '.mov': 'mov', '.mkv': 'matroska', '.webm': 'matroska'}
    ffmpeg = which('ffmpeg')

    def toggle_debug(_):
        debug_entries.clear()
        diagnostics.value = ('Debug is on. Click Refresh models or Send to capture a request.'
                             if debug.value else '')
        diagnostics.layout.display = '' if debug.value else 'none'

    def request_api(path, payload=None, *, timeout=10):
        data = json.dumps(payload).encode('utf-8') if payload is not None else None
        request = Request(f'http://127.0.0.1:11434/api/{path}', data=data,
                          headers={'Content-Type': 'application/json'})
        started = monotonic()
        http_status = None
        failure = ''
        try:
            with urlopen(request, timeout=timeout) as response:
                http_status = response.status
                result = json.load(response)
            if not isinstance(result, dict):
                raise ValueError('Ollama returned an invalid response.')
        except HTTPError as error:
            http_status = error.code
            try:
                with error:
                    body = json.load(error)
            except (OSError, ValueError):
                body = {}
            detail = body.get('error') if isinstance(body, dict) else None
            raise RuntimeError(detail or f'Ollama returned HTTP {error.code}.') from error
        except (OSError, ValueError, HTTPException, KeyboardInterrupt) as error:
            cause = error.reason if isinstance(error, URLError) else error
            failure = type(cause).__name__ if isinstance(cause, BaseException) else type(error).__name__
            raise
        finally:
            if debug.value:
                parts = [f'{request.get_method()} {request.full_url}']
                if payload and payload.get('model'):
                    parts.append(f'Model {payload["model"]}')
                if http_status is not None:
                    parts.append(f'HTTP {http_status}')
                if failure:
                    parts.append(failure)
                parts.append(f'{monotonic() - started:.2f}s elapsed (timeout {timeout}s)')
                debug_entries.append(' | '.join(parts))
                diagnostics.value = '<b>Recent Ollama requests</b><br>' + '<br>'.join(map(escape, debug_entries))
        return result

    def show_model():
        capabilities.clear()
        model_status.value = ''
        if not model.value:
            return
        try:
            info = request_api('show', {'model': model.value}, timeout=5)
        except api_errors:
            model_status.value = 'Model capabilities unavailable. Text files are supported.'
            return
        reported = info.get('capabilities')
        if isinstance(reported, list):
            capabilities.update(item.lower() for item in reported if isinstance(item, str))
        label = ', '.join(sorted(capabilities)) or 'unknown'
        model_status.value = f'Capabilities · {label}'
        if info.get('remote_host') or info.get('remote_model'):
            model_status.value += ' · Cloud model'

    def attachment_types():
        supported = set(text_types)
        if 'vision' in capabilities:
            supported.update(image_types)
            if ffmpeg:
                supported.update(video_formats)
        if 'audio' in capabilities:
            supported.update(audio_types)
        return supported

    def render_attachments():
        upload.accept = ','.join(sorted(attachment_types()))
        upload.disabled = busy or not model.value
        remove_files.disabled = busy or not pending_files
        if pending_files:
            attachment_status.value = 'Ready to send · ' + ', '.join(name for name, _, _ in pending_files)
        elif model.value:
            formats = ['UTF-8 text and code']
            if 'vision' in capabilities:
                formats.append('JPG, PNG, WebP')
                if ffmpeg:
                    formats.append('video frames from the first 8s')
                else:
                    formats.append('video needs FFmpeg')
            if 'audio' in capabilities:
                formats.append('WAV, MP3')
            attachment_status.value = ' · '.join(formats)
        else:
            attachment_status.value = ''

    def stage_files(change):
        uploads = change['new']
        if not uploads:
            return
        try:
            if busy or not model.value:
                return
            if len(uploads) > 8:
                raise ValueError('Choose up to 8 attachments per message.')
            entries = uploads.items() if isinstance(uploads, dict) else ((item['name'], item) for item in uploads)
            supported = attachment_types()
            files = []
            size = text_size = 0
            for name, item in entries:
                name = Path(name.replace(chr(92), '/')).name
                ext = Path(name).suffix.lower()
                if ext not in supported:
                    if ext in video_formats and 'vision' in capabilities and not ffmpeg:
                        raise ValueError('Video attachments need FFmpeg in the notebook runtime.')
                    raise ValueError(f'{name} uses a format unavailable for this model.')
                data = item['content']
                if not isinstance(data, (bytes, bytearray, memoryview)) or not data:
                    raise ValueError(f'{name} is empty or unreadable.')
                size += len(data)
                if size > 24 * 1024**2:
                    raise ValueError('Choose up to 24 MiB of attachments per message.')
                if ext in text_types:
                    text_size += len(data)
                    if text_size > 64 * 1024:
                        raise ValueError('Choose up to 64 KiB of text files per message.')
                files.append((name, ext, bytes(data)))
            pending_files[:] = files
            status.value = ''
        except (KeyError, TypeError, ValueError) as error:
            status.value = f'Attachment error. {error}'
        finally:
            if isinstance(uploads, dict):
                # ipywidgets 7 keeps a derived read-only value and separate synced buffers.
                with upload.hold_sync():
                    upload.data = []
                    upload.metadata = []
                    upload.set_trait('value', {})
            else:
                upload.value = ()
            render_attachments()

    def sample_video(ext, data):
        with TemporaryDirectory() as folder:
            path = Path(folder)
            clip = path / ('clip' + ext)
            clip.write_bytes(data)
            result = subprocess.run(
                [ffmpeg, '-nostdin', '-loglevel', 'error', '-protocol_whitelist', 'file',
                 '-f', video_formats[ext], '-threads', '2', '-i', str(clip), '-map', '0:v:0', '-an',
                 '-t', '8', '-vf', 'fps=1:start_time=0:round=up,scale=640:640:force_original_aspect_ratio=decrease',
                 '-frames:v', '8', '-threads', '1', str(path / 'frame-%02d.jpg')],
                stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True, errors='replace', timeout=30,
            )
            frames = sorted(path.glob('frame-*.jpg'))
            if result.returncode or not frames:
                detail = result.stderr.strip().splitlines()
                raise ValueError(f'Could not sample video frames. {detail[-1] if detail else "No frames decoded."}')
            return [frame.read_bytes() for frame in frames]

    def chat_message(content):
        parts = [content or 'Describe or summarize the attached files.']
        media = []
        supported = attachment_types()
        for name, ext, data in pending_files:
            if ext not in supported:
                raise ValueError(f'{name} uses a format unavailable for this model.')
            if ext in text_types:
                try:
                    text = data.decode('utf-8-sig')
                except UnicodeError as error:
                    raise ValueError(f'{name} must be UTF-8 text.') from error
                parts.append(f'Attached file {name}\n{text}\nEnd of attached file')
            elif ext in video_formats:
                frames = sample_video(ext, data)
                parts.append(f'Video {name} is represented by the next {len(frames)} sampled frames from its first 8 seconds.')
                media.extend(frames)
            else:
                if ext in audio_types:
                    valid = (data.startswith(b'RIFF') and data[8:12] == b'WAVE' if ext == '.wav' else
                             data.startswith(b'ID3') or (len(data) >= 2 and data[0] == 255 and data[1] & 224 == 224))
                else:
                    valid = {'jpg': data.startswith(b'\xff\xd8\xff'),
                             'png': data.startswith(b'\x89PNG\r\n\x1a\n'),
                             'webp': data.startswith(b'RIFF') and data[8:12] == b'WEBP'}[ext.lstrip('.').replace('jpeg', 'jpg')]
                if not valid:
                    raise ValueError(f'{name} does not match its file format.')
                media.append(data)
        message = {'role': 'user', 'content': '\n\n'.join(parts)}
        if media:
            # Ollama identifies WAV and MP3 bytes in this media array on audio-capable runners.
            message['images'] = [base64.b64encode(item).decode('ascii') for item in media]
        return message

    def format_turn_stats(selected, result):
        def count(values, key):
            value = values.get(key)
            return value if type(value) is int and value >= 0 else None

        prompt_tokens = count(result, 'prompt_eval_count')
        reply_tokens = count(result, 'eval_count')
        generation = count(result, 'eval_duration')
        parts = []
        if reply_tokens is not None and generation:
            parts.append(f'{reply_tokens * 1e9 / generation:.1f} tok/s')
        try:
            running = request_api('ps', timeout=3).get('models')
        except (*api_errors, KeyboardInterrupt):
            running = None
        active = {}
        if isinstance(running, list):
            active = next((entry for entry in running if isinstance(entry, dict)
                           and selected in (entry.get('name'), entry.get('model'))), {})
        context = count(active, 'context_length')
        if context and prompt_tokens is not None and reply_tokens is not None:
            used = prompt_tokens + reply_tokens
            parts.append(f'Context ~{min(100, 100 * used / context):.1f}% ({used:,} / {context:,} tokens)')
        else:
            parts.append('Context % unavailable')
        if prompt_tokens is not None:
            parts.append(f'{prompt_tokens:,} input tokens')
        if reply_tokens is not None:
            parts.append(f'{reply_tokens:,} output tokens')
        for field, label in [('total_duration', 'Total'), ('load_duration', 'Load')]:
            duration = count(result, field)
            if duration is not None:
                parts.append(f'{label} {duration / 1e9:.2f}s')
        vram = count(active, 'size_vram')
        if vram is not None:
            parts.append(f'{vram / 1024**3:.2f} GiB VRAM')
        reason = result.get('done_reason')
        if isinstance(reason, str) and reason:
            parts.append(f'Finished · {reason}')
        return ' | '.join(parts)

    def render():
        messages = transcripts.get(model.value, [])
        transcript.value = ''.join(
            f'<div style="margin:0.5em 0"><b>{role_labels[message["role"]]}</b>'
            f'<div style="white-space:pre-wrap;overflow-wrap:anywhere">{escape(message["content"])}</div></div>'
            for message in messages
        )
        model.disabled = refresh.disabled = debug.disabled = busy
        send.disabled = prompt.disabled = busy or not model.value
        clear.disabled = busy or not messages
        history = histories.get(model.value, [])
        compact.disabled = busy or sum(message['role'] == 'assistant' for message in history) < 2
        turn_status.value = turn_stats.get(model.value, '')
        render_attachments()

    role_labels = {'user': 'You', 'assistant': 'Model'}

    def refresh_models(_=None, *, tags=None):
        nonlocal refreshing
        selected = model.value
        refreshing = True
        try:
            if tags is None:
                models = request_api('tags').get('models', [])
                if not isinstance(models, list):
                    raise ValueError('Ollama returned an invalid model list.')
                names = (item.get('name') or item.get('model') for item in models if isinstance(item, dict))
                tags = sorted({name for name in names if isinstance(name, str) and name}, key=str.lower)
            model.options = tags
            model.value = selected if selected in model.options else next(iter(model.options), None)
            status.value = ('Select a model to chat.' if model.options
                            else 'No downloaded models found. Download a model and click Refresh models.')
        except api_errors as error:
            model.options = []
            model.value = None
            status.value = f'Could not list models. Make sure Ollama is running, then refresh. {error}'
        finally:
            refreshing = False
        if model.value != selected:
            pending_files.clear()
        show_model()
        render()

    def model_selected(_):
        if refreshing:
            return
        pending_files.clear()
        status.value = ''
        show_model()
        render()

    def ask_model(selected, messages, **options):
        result = request_api('chat', {'model': selected, 'messages': messages, 'stream': False, **options}, timeout=300)
        if result.get('error'):
            raise RuntimeError(result['error'])
        reply = result.get('message')
        answer = reply.get('content') if isinstance(reply, dict) else None
        if not isinstance(answer, str) or not answer.strip():
            raise ValueError('The model returned no text. Try a model that supports chat.')
        return result, {'role': 'assistant', 'content': answer}

    def send_message(_):
        nonlocal busy
        selected = model.value
        content = prompt.value.strip()
        if busy:
            return
        if not selected or not (content or pending_files):
            status.value = 'Select a model, then write a message or attach a file.'
            return

        busy = True
        render()
        status.value = f'Waiting for {selected}…'
        try:
            messages = [*histories.get(selected, []), chat_message(content)]
            result, reply = ask_model(selected, messages)
            histories[selected] = [*messages, reply]
            if pending_files:
                files = ', '.join(name for name, _, _ in pending_files)
                content = f'{content}\nAttachments · {files}'.strip()
            transcripts.setdefault(selected, []).extend(({'role': 'user', 'content': content}, reply))
            prompt.value = ''
            pending_files.clear()
            turn_stats[selected] = format_turn_stats(selected, result)
            status.value = ''
        except (*api_errors, subprocess.TimeoutExpired) as error:
            status.value = f'Chat failed for {selected}. {error}'
        except KeyboardInterrupt:
            status.value = 'Chat interrupted.'
        finally:
            busy = False
            render()

    def compact_chat(_):
        nonlocal busy
        selected = model.value
        if busy or compact.disabled:
            return
        history = histories[selected]
        instruction = {'role': 'user', 'content': (
            'Summarize the conversation above for use as chat history. Include user goals, decisions, '
            'important facts from attachments, and unresolved questions. Return only a concise summary.'
        )}
        busy = True
        render()
        status.value = 'Summarizing older turns…'
        try:
            result, reply = ask_model(selected, [*history[:-2], instruction], truncate=False,
                                      options={'num_predict': 1024})
            if result.get('done_reason') == 'length':
                raise ValueError('The summary was cut off.')
            summary = {'role': 'system', 'content': 'Reference summary of earlier chat turns.\n\n' + reply['content']}
            histories[selected] = [summary, *history[-2:]]
            turn_stats.pop(selected, None)
            status.value = 'Older turns compacted. Context usage updates after the next reply.'
        except api_errors as error:
            status.value = f'Could not compact chat. {error}'
        except KeyboardInterrupt:
            status.value = 'Compaction interrupted.'
        finally:
            busy = False
            render()

    def clear_chat(_):
        histories.pop(model.value, None)
        transcripts.pop(model.value, None)
        turn_stats.pop(model.value, None)
        status.value = 'Cleared the selected model’s chat.'
        render()

    def clear_files(_):
        pending_files.clear()
        status.value = ''
        render_attachments()

    model.observe(model_selected, names='value')
    debug.observe(toggle_debug, names='value')
    upload.observe(stage_files, names='value')
    refresh.on_click(refresh_models)
    clear.on_click(clear_chat)
    compact.on_click(compact_chat)
    remove_files.on_click(clear_files)
    send.on_click(send_message)
    toolbar = widgets.Layout(flex_flow='row wrap', align_items='center')
    display(widgets.VBox([model, widgets.HBox([refresh, clear, compact, debug], layout=toolbar),
                          model_status, turn_status, transcript, prompt,
                          widgets.HBox([upload, remove_files, send], layout=toolbar),
                          attachment_status, status, diagnostics]))
    refresh_models()
    return refresh_models


_refresh_downloaded_model_chat = open_downloaded_model_chat()

In [11]:
!ollama serve
!ollama list
!ollama ps
!tail -n 40 ollama.log

time=2026-11-26T08:28:05.219Z level=INFO source=routes.go:2484 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: LLAMA_ARG_FIT: LLAMA_ARG_FIT_TARGET: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_CREATE_REMOTE:false OLLAMA_DEBUG:INFO OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GO_TEMPLATE:true OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://0.0.0.0:11434 OLLAMA_IGPU_ENABLE: OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MAX_TRANSFER_STREAMS:4 OLLAMA_MODELS:/root/.ollama/models OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_NUM_PARALLEL:0 OLLAMA_ORIGINS:[* http://localhost https://localhost http://localhost:* https://localhost:* http://127.0.0.1 https://127.0.0.1 http://127.0.0.1:* https://127.0.0.1:* http://0.0.0.0 https:

Error: could not connect to ollama server, run 'ollama serve' to start it
Error: could not connect to ollama server, run 'ollama serve' to start it
print_info: f_attn_value_scale    = 0.0000
print_info: n_ff                  = 28672
print_info: n_expert              = 0
print_info: n_expert_used         = 0
print_info: n_expert_groups       = 0
print_info: n_group_used          = 0
print_info: causal attn           = 1
print_info: pooling type          = -1
print_info: rope type             = 0
print_info: rope scaling          = yarn
print_info: freq_base_train       = 1000000.0
print_info: freq_scale_train      = 0.015625
print_info: n_ctx_orig_yarn       = 4096
print_info: rope_yarn_log_mul     = 0.0000
print_info: rope_finetuned        = unknown
print_info: model type            = ?B
print_info: model params          = 125.03 B
print_info: general.name          = n/a
print_info: vocab type            = BPE
print_info: n_vocab               = 131072
print_info: n_merges             